# EdgeQuake — PhaseNet 台灣 CWA 微調 v2（Kaggle Dataset 版）

**v2 變更**：不再於 Kaggle 端下載大檔（2018 年檔案高達 88.6 GB，超出磁碟上限且
串流易斷）。改為掛載你在本機用 `make_cwa_train_subset.py` 做好的精簡訓練集
（2019 年，約 4 GB）。

**使用前設定：**
1. 右側 **Input → Add Input** → 選你上傳的 Dataset（含 metadata.csv + waveforms.hdf5）
2. Accelerator → **GPU（T4/P100）**
3. Internet → **On**（只為了抓 1 MB 的 pretrained 權重）
4. 確認能跑後 → **Save Version → Save & Run All (Commit)** 讓它在背景跑完

**協定註記**：官方切分為 ≤2018 訓練／2019 dev／≥2020 測試。因 ≤2018 檔案體積
限制，本訓練使用 2019（dev 年）之 90/10 切分，**2020–2021 測試集全程未接觸**。


In [ ]:
# Cell 1 — 安裝與設定
!pip install -q seisbench

CFG = dict(BATCH=256, EPOCHS=15, LR=1e-4, SIGMA=30, SEED=42)
import os, json, time, glob, random
import numpy as np, torch
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| torch", torch.__version__)

# 自動尋找掛載的訓練集
cands = glob.glob("/kaggle/input/**/metadata.csv", recursive=True)
print("input contents:", os.listdir("/kaggle/input"))
assert cands, "找不到掛載的 Dataset — 請在右側 Add Input 加入你的訓練集"
DATA_DIR = os.path.dirname(cands[0])
OUT_DIR = "/kaggle/working"
print("DATA_DIR =", DATA_DIR)


In [ ]:
# Cell 2 — 載入精簡訓練集（已是 100 Hz、已裁剪、含 train/dev split）
import seisbench.data as sbd

ds = sbd.WaveformDataset(DATA_DIR, sampling_rate=100, component_order="ZNE",
                          cache=None)
ds_train, ds_dev = ds.train(), ds.dev()
print(f"train {len(ds_train)} / dev {len(ds_dev)} traces")


In [ ]:
# Cell 3 — SeisBench 標準 PhaseNet 訓練管線
import seisbench.generate as sbg
import seisbench.models as sbm
from seisbench.util import worker_seeding
from torch.utils.data import DataLoader

phase_dict = {"trace_p_arrival_sample": "P", "trace_s_arrival_sample": "S"}

def clip_amplitude(state_dict):
    """CWA has dead/near-constant channels: per-window std-normalize can
    produce 1e9-scale values that permanently poison BatchNorm running
    stats (verified failure: in_bn.running_var reached 1.4e18 -> the
    model collapses to constant output in eval mode). Clip to +/-30 sigma."""
    x, meta = state_dict["X"]
    state_dict["X"] = (np.clip(x, -30.0, 30.0), meta)

def make_gen(dataset):
    gen = sbg.GenericGenerator(dataset)
    gen.add_augmentations([
        sbg.WindowAroundSample(list(phase_dict.keys()), samples_before=3000,
                               windowlen=6000, selection="random",
                               strategy="variable"),
        sbg.RandomWindow(windowlen=3001, strategy="pad"),
        sbg.Normalize(demean_axis=-1, amp_norm_axis=-1, amp_norm_type="std"),
        clip_amplitude,
        sbg.ChangeDtype(np.float32),
        sbg.ProbabilisticLabeller(label_columns=phase_dict,
                                  sigma=CFG["SIGMA"], dim=0),
    ])
    return gen

train_loader = DataLoader(make_gen(ds_train), batch_size=CFG["BATCH"],
                          shuffle=True, num_workers=4,
                          worker_init_fn=worker_seeding, drop_last=True,
                          pin_memory=True)
dev_loader = DataLoader(make_gen(ds_dev), batch_size=CFG["BATCH"],
                        shuffle=False, num_workers=2,
                        worker_init_fn=worker_seeding)

model = sbm.PhaseNet.from_pretrained("original").to(DEV)
print("labels:", model.labels)

# CRITICAL channel alignment: ProbabilisticLabeller outputs [P, S, Noise]
# but PhaseNet's channel order is model.labels (e.g. "NPS").
# Permute labels to the model order — without this the model is
# trained against scrambled targets (verified failure mode).
labeller_order = {"P": 0, "S": 1, "N": 2}
PERM = [labeller_order[ch] for ch in model.labels]
print("label permutation (labeller -> model):", PERM)

def loss_fn(y_pred, y_true, eps=1e-5):
    return -(y_true * torch.log(y_pred + eps)).sum(dim=1).mean()


In [ ]:
# Cell 4 — 微調（每 epoch 存驗證最佳）
# sanity check: with correct channel alignment the PRETRAINED model's
# initial loss must be small (~0.1-0.5). A value > 2 means scrambled labels.
model.eval()
with torch.no_grad():
    b0 = next(iter(dev_loader))
    l0 = loss_fn(model(b0["X"].to(DEV)), b0["y"][:, PERM].to(DEV))
print(f"initial (pretrained) dev loss: {float(l0):.4f}")
assert float(l0) < 2.0, "label/channel mismatch — do not train!"

opt = torch.optim.Adam(model.parameters(), lr=CFG["LR"])
best_val = float("inf"); history = []

def freeze_bn(m):
    """Fine-tuning: keep pretrained BatchNorm running stats frozen —
    standard practice, and immunizes against outlier-batch poisoning."""
    for mod in m.modules():
        if isinstance(mod, torch.nn.modules.batchnorm._BatchNorm):
            mod.eval()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    if train:
        freeze_bn(model)
    tot, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            x = batch["X"].to(DEV, non_blocking=True)
            y = batch["y"][:, PERM].to(DEV, non_blocking=True)
            pred = model(x)
            loss = loss_fn(pred, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.shape[0]; n += x.shape[0]
    return tot / max(n, 1)

for ep in range(1, CFG["EPOCHS"] + 1):
    t0 = time.time()
    tr = run_epoch(train_loader, True)
    va = run_epoch(dev_loader, False)
    history.append({"epoch": ep, "train_loss": tr, "val_loss": va})
    flag = ""
    if va < best_val:
        best_val = va
        torch.save({"state_dict": model.state_dict(), "labels": model.labels,
                    "base_weights": "original", "config": CFG,
                    "train_data": "CWA 2019 compact subset",
                    "epoch": ep, "val_loss": va},
                   f"{OUT_DIR}/phasenet_cwa_ft.pt")
        flag = "  <-- saved best"
    print(f"epoch {ep:02d}  train {tr:.4f}  val {va:.4f}  ({time.time()-t0:.0f}s){flag}")

json.dump(history, open(f"{OUT_DIR}/training_history.json", "w"), indent=2)
print("best val loss:", best_val)


In [ ]:
# Cell 5 — 驗證樣本快檢 + 收尾
import matplotlib.pyplot as plt
model.eval()
batch = next(iter(dev_loader))
with torch.no_grad():
    pred = model(batch["X"].to(DEV)).cpu().numpy()
lab = {c: j for j, c in enumerate(model.labels)}
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(batch["X"][0, 0], lw=0.6, color="#52514e"); axes[0].set_ylabel("Z")
axes[1].plot(pred[0, lab["P"]], color="#2a78d6", label="P")
axes[1].plot(pred[0, lab["S"]], color="#eb6834", label="S")
y_m = batch["y"][0][PERM]  # labeller order -> model order
axes[1].plot(y_m[lab["P"]], color="#2a78d6", ls="--", lw=0.8)
axes[1].plot(y_m[lab["S"]], color="#eb6834", ls="--", lw=0.8)
axes[1].legend(); axes[1].set_ylim(0, 1.05)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/val_sample.png", dpi=120); plt.show()
print("\n完成！右側 Output 下載 phasenet_cwa_ft.pt，回筆電執行：")
print("python scripts/eval_picking.py --dataset cwa --chunks _2020 _2021 "
      "--confirm-download --limit 1000 --state-dict outputs/phasenet_cwa_ft.pt")
